# SONAS — Colab kill-tests (month 0 → 1)

**Before running:** Runtime → Change runtime type → **A100 GPU** (T4 works but ~3× slower). Colab Pro+: keep background execution ON.

Run cells **top to bottom**. Cells 1–3 are setup (rerun each new session). The kill-test cells run in spec order:
**KT2 smoke → baseline → KT3 short → KT3 long → verdict.**

Resume rules after a disconnect: rerun Cells 1–3, then rerun the stage you were in. **KT3 stages skip already-finished models automatically.** KT2 and baseline restart from scratch (~1 GPU-hour each) — that is expected.

In [ ]:
# Cell 1 — Google Drive: journals + checkpoints survive disconnects
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/sonas"
import os; os.makedirs(WORK, exist_ok=True)
print("workdir:", WORK)

In [ ]:
# Cell 2 — clone the sonas-paper branch + install
!git clone --branch sonas-paper --depth 1 https://github.com/KarimAyman97/q1.git /content/ultralytics
%cd /content/ultralytics
!pip install -e . -q
!pip install -r research/sonas/requirements.txt -q
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Cell 3 — VisDrone download (first time ~15 min) + one-time COCO ground truth
from ultralytics.utils import DATASETS_DIR
from ultralytics.data.utils import check_det_dataset
from research.sonas.coco_eval import gt_to_coco

data = check_det_dataset("VisDrone.yaml")  # downloads + converts; moves files to VisDrone/{images,labels}/val
val_images = str(data["val"])
val_labels = val_images.replace("/images/", "/labels/")
gt_path = f"{WORK}/gt.json"
import os
if os.path.exists(gt_path):
    print("gt.json already exists, keeping it:", gt_path)
else:
    gt_to_coco(val_images, val_labels, gt_path, list(data["names"].values()))
    print("wrote", gt_path)

## KT2 — infrastructure smoke test (~1 GPU-hour)
8 random searched architectures each train 3 epochs. ✅ Expected final line: `KT2 PASSED (8 ran, 0 failed, 0 unrepairable)`.
If it prints FAILED — **stop here** and send the full cell output to Claude.

In [ ]:
!python -m research.sonas.scripts.kt2_smoke --workdir $WORK

## Baseline — stock yolo26n proxy (~1 GPU-hour)
Trains stock yolo26n for 15 epochs through the exact same evaluation path as search candidates. ✅ Done when it prints the baseline dict and writes `baseline_proxy.json` to Drive — **save the printed `map5095` number, the search needs it.**

In [ ]:
!python -m research.sonas.scripts.baseline_proxy --workdir $WORK

## KT3 short stage — 10 models × 15 epochs (~4–5 A100-hours, one evening)
Resume-safe: rerun after any disconnect, finished models are skipped.

In [ ]:
!python -m research.sonas.scripts.kt3_proxy_trust --workdir $WORK --stage short

## KT3 long stage — 10 models × 120 epochs (~40–50 A100-hours total)
The big one: several overnight background runs across days. Fully resume-safe — after every disconnect just rerun Cells 1–3 and this cell; it continues where it stopped.

In [ ]:
!python -m research.sonas.scripts.kt3_proxy_trust --workdir $WORK --stage long

## KT3 verdict — the life-or-death gate (instant)
Computes Kendall τ between the 15-epoch and 120-epoch rankings.

- **τ ≥ 0.6 → PROXY TRUSTED** — send Claude the τ number + the baseline `map5095`; the evolution search gets built next.
- **τ < 0.6 → NOT TRUSTED** — stop, send Claude the output; the proxy gets lengthened before any search money is spent.

In [ ]:
!python -m research.sonas.scripts.kt3_proxy_trust --workdir $WORK --stage verdict

---
# 📘 Chapter 2 — THE SEARCH (~170 auditions, ~70 A100-hours over ~2 weeks)

**Self-gated:** this cell REFUSES to start unless the KT3 verdict passes (τ ≥ 0.6, full panel) and the baseline exists — it reads both from Drive automatically, nothing to type in.

NSGA-II evolution: population 20 × 8 generations of 15-epoch auditions. **Fully resume-safe** — every finished audition is journaled to `search.ndjson` on Drive the moment it completes; after any disconnect rerun Cells 1–3, then this cell, and it continues exactly where it stopped (finished models are never re-trained).

When it finishes it writes **`pareto_front.json`** to Drive and prints the menu of winners: for each, its cost (GFLOPs/params) and its small-object score. **Send that printout (or the json) to Claude** → winner selection + full training (Chapter 3) get built next.

In [ ]:
!python -m research.sonas.scripts.campaign --workdir $WORK